# Phase 2 — Preprocessing & Feature Engineering

Goals:
1. Clean the data (handle nulls, normalise strings)
2. Encode `ingredients` → Multi-hot vector with `MultiLabelBinarizer`
3. Encode `family`, `subfamily`, `gender` → One-hot vectors with `OneHotEncoder`
4. Concatenate into a final feature matrix `(N, F)`
5. Save all encoders and the matrix to `../models/`

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import joblib
import os
from scipy.sparse import hstack, csr_matrix, save_npz
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from data_loader import load_data

MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)

df = load_data()
print(f'Loaded {len(df):,} perfumes')

## Step 1 — Clean & Normalise

In [ ]:
def clean_string(val, default='Unknown'):
    if pd.isna(val) or str(val).strip() == '':
        return default
    return str(val).strip().upper()

def clean_ingredients(val):
    if not isinstance(val, list):
        return []
    return [str(i).strip().title() for i in val if str(i).strip()]

df['family']      = df['family'].apply(lambda x: clean_string(x, 'UNKNOWN'))
df['subfamily']   = df['subfamily'].apply(lambda x: clean_string(x, 'UNKNOWN'))
df['gender']      = df['gender'].apply(lambda x: clean_string(x, 'UNKNOWN'))
df['brand']       = df['brand'].apply(lambda x: clean_string(x, 'UNKNOWN'))
df['ingredients'] = df['ingredients'].apply(clean_ingredients)

# Drop perfumes with no ingredients (can't recommend meaningfully)
before = len(df)
df = df[df['ingredients'].apply(len) > 0].reset_index(drop=True)
print(f'Dropped {before - len(df)} perfumes with empty ingredients')
print(f'Remaining: {len(df):,}')

# Save cleaned dataframe for use in the app
joblib.dump(df, f'{MODELS_DIR}/perfume_df.pkl')
print('Saved perfume_df.pkl')

## Step 2 — Encode Ingredients (Multi-hot)

In [ ]:
mlb = MultiLabelBinarizer(sparse_output=True)
ingredients_matrix = mlb.fit_transform(df['ingredients'])

print(f'Ingredients vocabulary size : {len(mlb.classes_):,}')
print(f'Ingredients matrix shape    : {ingredients_matrix.shape}')
print(f'Sparsity                    : {1 - ingredients_matrix.nnz / (ingredients_matrix.shape[0] * ingredients_matrix.shape[1]):.3f}')

joblib.dump(mlb, f'{MODELS_DIR}/mlb_ingredients.pkl')
print('Saved mlb_ingredients.pkl')

## Step 3 — Encode Family, Subfamily, Gender (One-hot)

In [ ]:
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
cat_features = df[['family', 'subfamily', 'gender']]
cat_matrix = ohe.fit_transform(cat_features)

print(f'Categorical features: {cat_matrix.shape[1]} dimensions')
print(f'  family categories   : {len(ohe.categories_[0])}')
print(f'  subfamily categories: {len(ohe.categories_[1])}')
print(f'  gender categories   : {len(ohe.categories_[2])}')

joblib.dump(ohe, f'{MODELS_DIR}/ohe_categories.pkl')
print('Saved ohe_categories.pkl')

## Step 4 — Build Feature Matrices for All 4 Approaches

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Approach A: Ingredients only
matrix_A = ingredients_matrix

# Approach B: Ingredients + OHE(family, subfamily, gender)  [RECOMMENDED]
matrix_B = hstack([ingredients_matrix, cat_matrix])

# Approach C: B + TF-IDF on description
tfidf = TfidfVectorizer(max_features=500, stop_words='english')
desc_matrix = tfidf.fit_transform(df['description'].fillna(''))
matrix_C = hstack([ingredients_matrix, cat_matrix, desc_matrix])
joblib.dump(tfidf, f'{MODELS_DIR}/tfidf_description.pkl')

# Approach D: Weighted — ingredients x2, categories x1 (B with ingredient emphasis)
matrix_D = hstack([ingredients_matrix * 2, cat_matrix])

for name, mat in [('A', matrix_A), ('B', matrix_B), ('C', matrix_C), ('D', matrix_D)]:
    save_npz(f'{MODELS_DIR}/matrix_{name}.npz', mat.tocsr())
    print(f'Approach {name}: shape={mat.shape}, nnz={mat.nnz:,}')

print('\nAll matrices saved.')

## Step 5 — Verify Encoding with a Sample

In [ ]:
sample = df.iloc[0]
print('Sample perfume:')
print(f'  Name       : {sample["name_perfume"]}')
print(f'  Brand      : {sample["brand"]}')
print(f'  Gender     : {sample["gender"]}')
print(f'  Family     : {sample["family"]}')
print(f'  Subfamily  : {sample["subfamily"]}')
print(f'  Ingredients: {sample["ingredients"][:5]}...')
print(f'\nRow vector B shape  : {matrix_B[0].shape}')
print(f'Non-zero entries    : {matrix_B[0].nnz}')